[![Open in GitHub Codespaces](https://github.com/codespaces/badge.svg)](https://github.com/codespaces)


<!-- gen:values.yml#weeks.03.notebook-title|Introduction -->
# Week 3: Assessment briefing
<!-- /gen -->

This is a transition week. There is no [Quest](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-quest), no [Debrief](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-debrief), and no new Python. The session <!-- plain-language: everyday brief -->briefs you on **both assessments**. Then you get workshop time to write your first [AI Use Record](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-ai-use-record) entry and start on the individual report. Read this notebook before the session if you can. Refer back to it afterwards.

This week's materials run in your **GitHub Codespace**.

## Check your Codespace

You write no code this week, but you run a few short cells. This first one prints the versions of [Python](https://docs.python.org/3/library/platform.html) and [pandas](https://pandas.pydata.org/docs/) your Codespace runs, and the pandas version the module expects. Run it with the ▶ button or Shift+Enter. If the two pandas versions differ, tell your workshop tutor.

The cell also finds the module's folder, the one that contains `Tools/`, by looking in this folder and in each folder above it. That way the notebook works where it is and from your own copy in `work/Week/03/`.

In [ ]:
# Print the Python and pandas versions, and the pandas version the module expects
import platform
import re
from pathlib import Path

import pandas as pd

print("Python:", platform.python_version())
print("pandas:", pd.__version__)


def find(*parts):
    """Return the first folder, from here upwards, that contains the path *parts*."""
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        for place in (folder, folder / "MN5813" / "2026-27"):  # The second is for a copy in work/
            if place.joinpath(*parts).exists():
                return place
    return None


module = find("Tools", "fetch_abcd.py")  # The folder that contains Tools/ and Week/
if module and (module / "requirements.lock").is_file():
    lock = module / "requirements.lock"  # The module's list of exact versions
    expected = re.search(r"^pandas==(\S+)", lock.read_text(), re.MULTILINE).group(1)
    print("pandas the module expects:", expected)

## The assessments

Only the individual report counts towards your module mark:

- The [group project](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Assessment/group-project.md): One notebook report per team on the London Fire Brigade records, plus the team's process record. It is formative: Your team gets a feedback letter, not a mark
- The [individual report](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Assessment/individual-report.md): One notebook on the abcd project record, plus a PDF and your [SpecStory](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-specstory) record. It is your module mark

Each linked page sets out its dataset, deliverables, and timeline. This briefing is about *how each criterion is evidenced*.

### The group project (formative)

In teams, you take on **Engagement FRS-2026**. The operations director of a fire and rescue service wants a story she can take to her board. The data is the London Fire Brigade's own record of every 999 incident it attended and every engine it sent.

The [group project brief](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Assessment/group-project.md) sets out the task, the requirements, the deliverables, the team size, and the timeline. The [data note](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Assessment/lfb-data.md) explains how your team fetches the tables. It also explains the challenges in the data.

The consultancy recommends that your team meets on Microsoft Teams from Week 4, with a [Stoa](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-stoa) room as the alternative. The [Stoa workflow guide](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/stoa-workflow.md) explains how that works. Using Stoa is optional. We will have a quick test run next week.

Here is an early look at that data. The cell below uses the [requests](https://requests.readthedocs.io/) library to read the first 1,000 rows of the Brigade's mobilisation file, one row per engine sent. It prints five of them. You fetch the full tables in the Week 4 workshop.

In [ ]:
# Read the first 1,000 rows of the live mobilisation file, and print five
import csv
import sys
from pathlib import Path

import requests

lines = []  # The header line, then one line per engine sent
if module is None:  # Found by the first code cell
    print("Run the first code cell, then this one, in your Codespace to see the live rows.")
else:
    sys.path.insert(0, str(module / "Tools"))
    import fetch_lfb  # The script that contains the file's address

    url = fetch_lfb.SOURCES["mobilisations"]["url"]
    try:
        with requests.get(url, stream=True, timeout=30) as response:
            response.raise_for_status()
            response.encoding = "utf-8-sig"  # The file starts with a byte-order mark
            for line in response.iter_lines(decode_unicode=True):
                lines.append(line)
                if len(lines) == 1001:  # The header line and 1,000 rows
                    break
    except requests.RequestException as error:
        print("Could not reach the London Datastore:", error)
    for row in list(csv.DictReader(lines))[:5]:
        print(f"{row['DateAndTimeMobilised']}  {row['BoroughName']}: An engine from "
              f"{row['DeployedFromStation_Name']} arrived after {row['AttendanceTimeSeconds']} seconds")

### A preview: Plain Python and pandas

From Week 4 you work with [pandas](https://pandas.pydata.org/docs/), Python's library for tables. The next two cells ask those 1,000 rows one question: Which five boroughs received the most engines, and how quickly did engines arrive there? This first cell uses plain Python, counting and adding up in dictionaries.

In [ ]:
# Plain Python: Count the engines per borough, and average their attendance times
import time

if not lines:
    print("Run the cell above first, in your Codespace.")
else:
    start = time.perf_counter()  # Start the clock
    engines, seconds, timed = {}, {}, {}
    for row in csv.DictReader(lines):
        borough = row["BoroughName"].title()
        engines[borough] = engines.get(borough, 0) + 1
        if row["AttendanceTimeSeconds"] not in ("", "NULL"):  # Some engines have no time
            seconds[borough] = seconds.get(borough, 0) + int(row["AttendanceTimeSeconds"])
            timed[borough] = timed.get(borough, 0) + 1
    busiest = sorted(engines, key=lambda b: (-engines[b], b))[:5]
    elapsed = time.perf_counter() - start  # Stop the clock

    print(f"{'borough':<24}{'engines':>8}{'avg_minutes':>13}")
    for borough in busiest:
        average = seconds.get(borough, 0) / timed[borough] / 60 if timed.get(borough) else float("nan")
        print(f"{borough:<24}{engines[borough]:>8}{average:>13.1f}")
    print(f"\nPlain Python took {elapsed * 1000:.1f} milliseconds")

The cell below asks [pandas](https://pandas.pydata.org/docs/user_guide/10min.html) the same question. It loads the rows into a table, then groups and summarises them with [groupby](https://pandas.pydata.org/docs/user_guide/groupby.html). Both cells time themselves with [time.perf_counter](https://docs.python.org/3/library/time.html#time.perf_counter). At this size, both take a few milliseconds. Compare how much code each needs instead.

In [ ]:
# pandas: The same question, answered and shown as a table
import io
import time

import pandas as pd

if lines:
    start = time.perf_counter()  # Start the clock
    sample = pd.read_csv(io.StringIO("\n".join(lines)), na_values=["NULL"])
    sample["BoroughName"] = sample["BoroughName"].str.title()
    table = (sample.groupby("BoroughName")
             .agg(engines=("IncidentNumber", "size"), avg_minutes=("AttendanceTimeSeconds", "mean"))
             .sort_values("engines", ascending=False, kind="stable")
             .head(5))
    table["avg_minutes"] = (table["avg_minutes"] / 60).round(1)
    elapsed = time.perf_counter() - start  # Stop the clock

    display(table)
    print(f"pandas took {elapsed * 1000:.1f} milliseconds")
else:
    print("Run the cell above first, in your Codespace.")

### The main assessment: An individual report

You are the lead [Analyst](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-analyst) on your own client account. The dataset is the **abcd project record**, the documented history of a small open-source tool. `abcd` is my own project. Treat me as your client. I expect you to report findings that are critical of the project.

In Week 6 you choose one of the **[briefs](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-brief)**. Each [brief](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-brief) has a stakeholder and a starting question. Sharpen that starting question into a business question the stakeholder needs answered.

The [individual report brief](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Assessment/individual-report.md) sets out the dataset rules, the required structure, and the timeline. The [record primer](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Assessment/abcd-record.md) explains [the record](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-abcd-record) and how you fetch it. The [briefs page](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Assessment/individual-briefs.md) lists the briefs.

The [Template.ipynb](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Assessment/Template.ipynb) in the Assessment folder is the starting point for the report. Skim it this week.

Now fetch the abcd record itself. The cell below runs `Tools/fetch_abcd.py` with Python's [subprocess](https://docs.python.org/3/library/subprocess.html) module, as the [record primer](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Assessment/abcd-record.md) describes. The first run downloads a copy of the project. Then it prints each table it writes into `work/data/abcd/`, with its number of rows. The `work/` folder is your own, so the tables stay with the rest of your work.

In [ ]:
# Run the fetch script from the folder that contains Tools/, and show what it prints
import subprocess
import sys

if module is None:  # Found by the first code cell
    print("Run the first code cell, then this one, in your Codespace to fetch the record.")
else:
    result = subprocess.run([sys.executable, "Tools/fetch_abcd.py"], cwd=module,
                            capture_output=True, text=True)
    print(result.stdout, result.stderr)
    place = find("work")  # The folder that contains your work folder, found the same way
    if place:
        print("The tables are in", place / "work" / "data" / "abcd")

## Read the interpretation document

The [assessment interpretation](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Assessment/interpretation.md) explains how markers read the rubrics in an AI-open module, and what both assessments share.

**Read it in full before Week 4.** I walk you through it in the session, but the walk-through does not replace reading it.

## Key changes

I've mentioned it before in class: This year's materials differ from those we used previously. Below are the main changes. <!-- docs-lint: allow: this section describes what changed from last year -->

### Change 1: AI-open, with an AI Use Record

You may use AI for every part of both assessments, and I expect you to. Nobody here asks whether you used AI. You show *how* you used it. Markers assess your judgement.

Every submission, group and individual, ends with an **AI Use Record**. The [interpretation document](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Assessment/interpretation.md) sets out the AI-open statement and what the record must contain. It also gives a worked example. The [AI guide](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/ai-guide.md) shows the working practice behind it.

### Change 2: The group process is recorded

For the group project, your team's recorded working process is evidence for your feedback letter. The [group project brief](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Assessment/group-project.md) explains how to record it, with or without [Stoa](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-stoa).

For **Data management**, markers use your <!-- plain-language: everyday decision -->decision trail as the main evidence that your procedures are appropriate.

I don't read your team's record during term. I read it after you submit, as evidence **in your favour**.

### Change 3: Design rationales and TRACE for every key figure

Choosing which chart to make is your job.

Checking that it is clear, accessible, and tells "the truth about the data" (Tufte) is also your job. Markers assess that judgement by reading every key figure in both assessments against the **[TRACE checklist](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-trace)**. Each figure must carry a short **[design rationale](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-design-rationale)**.

The [individual report brief](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Assessment/individual-report.md) sets out what a rationale contains and how long it is. Weeks 6, 8, and 9 build this "craft". We'll talk about that word much more in later sessions. [Template.ipynb](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Assessment/Template.ipynb) contains rationale placeholders.